# Lab 10 Prelab Walkthrough — Peaks, Log Decrement, and Exponential Fits

Work through this notebook **before** your Lab 10 session. It teaches the
*new* Python skills on simulated signals — the same math your rig and motor
will produce for real. Report the **CHECKPOINT** values in the Prelab 10
quiz on Canvas.

Everything here runs without hardware.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.signal import find_peaks

rng = np.random.default_rng(42)      # seeded: everyone gets the same data

## Skill 1 — `find_peaks`: locating oscillation maxima

First, manufacture a ringdown like the one your tap test will record: a
decaying sine plus noise.

In [ ]:
fs = 1000
t = np.arange(0, 2.0, 1/fs)
fn_true, zeta_true = 35.0, 0.04      # the "truth" we'll try to recover

wn = 2*np.pi*fn_true
wd = wn*np.sqrt(1 - zeta_true**2)
ring = 0.30*np.exp(-zeta_true*wn*t)*np.sin(wd*t)
y = ring + rng.normal(0, 0.003, t.size)

plt.plot(t, y, lw=0.8)
plt.xlabel('Time (s)'); plt.ylabel('Signal (V)'); plt.show()

In [ ]:
peaks, props = find_peaks(y, height=0.02, distance=15)
peaks = peaks[:10]                   # keep the first 10 clean peaks
print('peak times:', t[peaks].round(4))
print('peak amps :', y[peaks].round(4))

plt.plot(t, y, lw=0.8)
plt.plot(t[peaks], y[peaks], 'ro')
plt.xlim(0, 0.6); plt.xlabel('Time (s)'); plt.show()

The two keyword arguments are *designed against the signal*:

- `height=0.02` — real peaks must clear ~7× the noise σ (0.003 V), so
  noise bumps never qualify;
- `distance=15` — no two peaks within 15 samples; the damped period is
  ~29 samples here, so a single physical peak can't get double-counted.

**CHECKPOINT 1:** Report the time of the **third** detected peak
(`t[peaks][2]`) to 4 decimals.

## Skill 2 — Log decrement: damping from peak amplitudes

Successive peaks of a damped oscillation shrink by a constant ratio.
The average log of that ratio is the **log decrement** δ, and

$$\zeta = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}}$$

In [ ]:
Ap = y[peaks]
tp = t[peaks]

delta = np.mean(np.log(Ap[:-1] / Ap[1:]))    # every successive-pair ratio
zeta = delta / np.sqrt(4*np.pi**2 + delta**2)

Td = np.diff(tp).mean()                      # damped period
omega_d = 2*np.pi / Td
omega_n = omega_d / np.sqrt(1 - zeta**2)

print(f"zeta = {zeta:.4f}   (truth: {zeta_true})")
print(f"fn   = {omega_n/2/np.pi:.2f} Hz (truth: {fn_true})")

The slicing pair `Ap[:-1] / Ap[1:]` divides each peak by its successor —
nine ratios from ten peaks, all at once.

**CHECKPOINT 2:** Report your computed `zeta` to 4 decimals.

## Skill 3 — Linearizing an exponential: τ by `polyfit`

A first-order step response approaches its final value exponentially.
Normalize it as $y = (T_{ss}-T)/(T_{ss}-T_0)$ and $\ln y$ is a straight
line with slope $-1/\tau$ — turning time-constant extraction into the
`np.polyfit` you've used since Lab 01.

In [ ]:
tau_true = 0.060
t2s = np.arange(0, 1.0, 1/fs)
T0, Tss = 0.40, 1.40                     # newtons, like your step data
T = Tss - (Tss - T0)*np.exp(-t2s/tau_true) + rng.normal(0, 0.01, t2s.size)

y = (Tss - T) / (Tss - T0)               # normalized: starts 1, decays to 0
mask = (t2s < 0.30) & (y > 0.10) & (y < 0.90)

pc = np.polyfit(t2s[mask], np.log(y[mask]), 1)
tau = -1 / pc[0]
print(f"tau = {tau*1000:.1f} ms   (truth: {tau_true*1000:.0f} ms)")

plt.plot(t2s[mask], np.log(y[mask]), 'r.', ms=3)
plt.plot(t2s[mask], np.polyval(pc, t2s[mask]), 'b-')
plt.xlabel('Time (s)'); plt.ylabel('ln y'); plt.show()

Study the `mask` — it is *designed*, not decorative: `y > 0.10` and
`y < 0.90` keep the clean middle of the decay (both ends are
noise-dominated), and `t2s < 0.30` stops settled noise that randomly pokes
above 0.10 from sneaking into the fit late and wrecking the slope.

**CHECKPOINT 3:** Report your fitted `tau` in ms to 1 decimal.

## Skill 4 — Block averaging: honest CIs for time series

5000 samples of a drifting signal are *not* 5000 independent measurements.
Cut the record into 1-second blocks; the block means are the honest
sample.

In [ ]:
drift = 0.02*np.sin(2*np.pi*0.3*np.arange(5000)/1000)      # slow wander
sig = 5.0 + drift + rng.normal(0, 0.05, 5000)

n_blk = len(sig) // 1000
blocks = sig[:n_blk*1000].reshape(n_blk, 1000).mean(axis=1)

naive_CI = stats.t.ppf(0.975, df=4999) * sig.std(ddof=1)/np.sqrt(5000)
block_CI = stats.t.ppf(0.975, df=n_blk-1) * blocks.std(ddof=1)/np.sqrt(n_blk)

print(f"naive CI: ±{naive_CI:.4f}   block CI: ±{block_CI:.4f}")
print("the honest interval is wider — because the data really does wander")

`reshape(n_blk, 1000)` folds the 1-D record into a (5 × 1000) array —
one row per second — so `.mean(axis=1)` collapses each row to its block
mean. (Compare `axis=0` from Lab 09: axes are directions, reductions
collapse them.)

**CHECKPOINT 4:** Report the block CI (`block_CI`) to 4 decimals.

## Summary — where each skill appears in lab

| Skill | Where you'll use it in Lab 10 |
|-------|-------------------------------|
| `find_peaks` | Part-2: locating tap-test ringdown peaks |
| log decrement | Part-2: rig ζ and fn from those peaks |
| linearized exponential fit | Part-3: motor time constant from step data |
| block averaging | Part-5: honest CI on the hover thrust mean |
